# IoTShield — 03: Klasik baseline modeller
**SE3513 · ASEADOS-SDN-IoT**

Soru: Klasik modeller (lojistik regresyon, karar ağacı) final modelle (Set B: LightGBM, F_full) aynı katlarda ölçüldüğünde ne skor veriyor?
Bu notebook #7'deki baseline çalışmasının yeni düzene taşınmış hâli:
- Eski train/test bölmesi yerine oturum gruplu 3 kat (`get_folds`), sonuçlar `run_cv` ile (Kural 1, 2).
- Hiperparametre araması yok, iki model de varsayılan parametrelerle (Kural 1).
- Sütun seti **F_full**: final modelle karşılaştırılabilir olsun diye. (LR, F_timing ile WP-D3'te ölçüldü: 0,352 ± 0,057.)
- Her model iki görevde: 5 sınıf (`Label`) ve Benign / Attack (`label_binary`).

## 0. Kurulum

In [1]:
import os
import sys
import numpy as np, pandas as pd

sys.path.append('..')
from src.config import DATA_PATH, RANDOM_STATE, F_FULL
from src.preprocessing import clean, NO_CLIP_COL
from src.cv import get_folds, print_fold_table, run_cv
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

pd.set_option('display.max_columns', 30)
BASELINE_CSV = '../reports/results_baseline.csv'
SHOW_COLS = ['experiment', 'model', 'feature_set', 'macro_f1_mean', 'macro_f1_std', 'macro_f1_clean',
             'macro_f1_broken', 'recall_Dos', 'recall_DDoS', 'recall_Bot', 'recall_Prob', 'bot_ddos_confusion',
             'fit_time_s', 'note']

## 1. Veri ve katlar
`clean()` verisi; katlar bir kez alınır ve bütün deneylerde aynı katlar kullanılır. Sonuç dosyası her çalıştırmada sıfırdan yazılır (`run_cv` satır ekler; tekrar çalıştırmada çift satır olmasın).

In [2]:
data = clean(pd.read_csv(DATA_PATH, low_memory=False))
print('veri:', data.shape)
print(data['Label'].value_counts())

if os.path.exists(BASELINE_CSV):
    os.remove(BASELINE_CSV)

folds = get_folds(data)
fold_table = print_fold_table(data, folds)
results = []

clean: dropped 70 rows with inf/NaN
clean: clipped negative values to 0 in 427 rows


clean: removed 29930 duplicate rows (features + label)
clean: final shape (427044, 63)
veri: (427044, 63)
Label
Benign    233747
Dos       126506
DDoS       50269
Bot         8957
Prob        7565
Name: count, dtype: int64


Test kısmındaki satır sayıları (sınıf × kat):
        fold_0  fold_1  fold_2
Label                         
Benign   76910   79965   76872
Bot       1757    2585    4615
DDoS     23953   10599   15717
Dos      16482   16711   93313
Prob      2803    2316    2446
Test kısmındaki oturum sayıları (sınıf × kat):
        fold_0  fold_1  fold_2
Label                         
Benign       9       3      11
Bot          1       1       1
DDoS         1       1       2
Dos         19       3       1
Prob         1       1       4


## 2. Lojistik regresyon (LR)
Pipeline WP-D3 ile aynı: `log1p → StandardScaler → LogisticRegression(max_iter=1000)`. Tek fark: F_full'daki `Init Bwd Win Byts` sütununda -1 ("bilgi yok", `clean()` bunu bilerek kırpmıyor) var ve `log1p(-1) = -inf`. Bu yüzden sadece bu sütun önce +1 kaydırılır (-1 → 0). Log ve ölçekleme Pipeline içinde, yani sadece eğitim katında fit edilir (Kural 4).
#7'den farkı: `class_weight` ve ayrı doğrulama seti yok; sadece varsayılan model.

Not (Kural 5 istisnası): deney modeli normalde LightGBM; bu notebook'un amacı klasik modelleri ölçmek.

In [3]:
LR_MAX_ITER = 1000


def log_transform(X):
    """Bütün sütunlara log1p uygular; Init Bwd Win Byts önce +1 kaydırılır. Yeni DataFrame döndürür."""
    X = X.copy()
    # Bu sütunda -1 = "bilgi yok" (bozuk satırların neredeyse tamamı); log1p(-1) = -inf olmasın diye -1 -> 0.
    X[NO_CLIP_COL] = X[NO_CLIP_COL] + 1
    return np.log1p(X)


def lr_pipeline():
    """log_transform -> StandardScaler -> LogisticRegression, imblearn Pipeline içinde (sadece eğitim katında fit)."""
    return Pipeline([('log', FunctionTransformer(log_transform)),
                     ('scale', StandardScaler()),
                     ('model', LogisticRegression(max_iter=LR_MAX_ITER))])


print('=== base-LR-5class ===')
row, cm, oof = run_cv(lr_pipeline(), data, F_FULL, BASELINE_CSV, experiment='base-LR-5class', sampler='none',
                      feature_set='F_full', folds=folds, note='log1p (Init Bwd Win Byts +1)+StandardScaler+LR, default parameters')
results.append(row)
display(cm)

print('=== base-LR-binary ===')
row, cm, oof = run_cv(lr_pipeline(), data, F_FULL, BASELINE_CSV, experiment='base-LR-binary', sampler='none',
                      feature_set='F_full', folds=folds, target='label_binary',
                      note='binary task; log1p (Init Bwd Win Byts +1)+StandardScaler+LR, default parameters')
results.append(row)

display(pd.DataFrame(results)[SHOW_COLS].round(3))

=== base-LR-5class ===
Kat 0


  eğitim: 304139 satır (çelişki atılan: 1000 ) | test: 121905 | fit: 49.3 sn


Kat 1


  eğitim: 314008 satır (çelişki atılan: 860 ) | test: 112176 | fit: 33.7 sn
Kat 2


  eğitim: 233464 satır (çelişki atılan: 617 ) | test: 192963 | fit: 27.5 sn


Kat F1'leri: [0.394  0.4179 0.459 ] → 0.4237 ± 0.0329
CSV'ye eklendi: ../reports/results_baseline.csv


... (3 outputs left out)

  eğitim: 314395 satır (çelişki atılan: 473 ) | test: 112176 | fit: 4.5 sn
Kat 2


  eğitim: 233667 satır (çelişki atılan: 414 ) | test: 192963 | fit: 4.9 sn


Kat F1'leri: [0.7502 0.7817 0.8058] → 0.7792 ± 0.0279
CSV'ye eklendi: ../reports/results_baseline.csv


,experiment,model,feature_set,macro_f1_mean,macro_f1_std,macro_f1_clean,macro_f1_broken,recall_Dos,recall_DDoS,recall_Bot,recall_Prob,bot_ddos_confusion,fit_time_s,note
0,base-LR-5class,LogisticRegression,F_full,0.424,0.033,0.804,0.278,0.506,0.72,0.0,0.192,0.138,36.831,log1p (Init Bwd Win Byts +1)+StandardScaler+LR...
1,base-LR-binary,LogisticRegression,F_full,0.779,0.028,0.805,0.580,NaN,NaN,NaN,NaN,NaN,6.195,binary task; log1p (Init Bwd Win Byts +1)+Stan...


## 3. Karar ağacı (Decision Tree)
`DecisionTreeClassifier(random_state=42)`, varsayılan parametreler. Ağaç ölçekten etkilenmediği için log/ölçekleme yok.
#7'den farkı: `RandomizedSearchCV` ve özel sınıf ağırlıkları yok (Kural 1: hiperparametre araması yok).

In [4]:
def dt_pipeline():
    """Sadece DecisionTreeClassifier içeren imblearn Pipeline (run_cv son adımın adını model sütununa yazar)."""
    return Pipeline([('model', DecisionTreeClassifier(random_state=RANDOM_STATE))])


print('=== base-DT-5class ===')
row, cm, oof = run_cv(dt_pipeline(), data, F_FULL, BASELINE_CSV, experiment='base-DT-5class', sampler='none',
                      feature_set='F_full', folds=folds, note='DecisionTree, default parameters')
results.append(row)
display(cm)

print('=== base-DT-binary ===')
row, cm, oof = run_cv(dt_pipeline(), data, F_FULL, BASELINE_CSV, experiment='base-DT-binary', sampler='none',
                      feature_set='F_full', folds=folds, target='label_binary',
                      note='binary task; DecisionTree, default parameters')
results.append(row)

display(pd.DataFrame(results)[SHOW_COLS].round(3))

=== base-DT-5class ===
Kat 0


  eğitim: 304139 satır (çelişki atılan: 1000 ) | test: 121905 | fit: 3.6 sn
Kat 1


  eğitim: 314008 satır (çelişki atılan: 860 ) | test: 112176 | fit: 4.3 sn
Kat 2


  eğitim: 233464 satır (çelişki atılan: 617 ) | test: 192963 | fit: 3.0 sn


Kat F1'leri: [0.4064 0.4771 0.3836] → 0.4224 ± 0.0487
CSV'ye eklendi: ../reports/results_baseline.csv


,Benign,Dos,DDoS,Bot,Prob
Benign,211586,6532,12042,1648,1939
Dos,84824,34170,6188,719,605
DDoS,13393,6136,23842,4603,2295
Bot,1589,711,5068,1258,331
Prob,1954,646,2214,311,2440


... (2 outputs left out)

  eğitim: 314395 satır (çelişki atılan: 473 ) | test: 112176 | fit: 3.8 sn
Kat 2


  eğitim: 233667 satır (çelişki atılan: 414 ) | test: 192963 | fit: 2.9 sn


Kat F1'leri: [0.7153 0.8583 0.6257] → 0.7331 ± 0.1173
CSV'ye eklendi: ../reports/results_baseline.csv


,experiment,model,feature_set,macro_f1_mean,macro_f1_std,macro_f1_clean,macro_f1_broken,recall_Dos,recall_DDoS,recall_Bot,recall_Prob,bot_ddos_confusion,fit_time_s,note
0,base-LR-5class,LogisticRegression,F_full,0.424,0.033,0.804,0.278,0.506,0.720,0.00,0.192,0.138,36.831,log1p (Init Bwd Win Byts +1)+StandardScaler+LR...
1,base-LR-binary,LogisticRegression,F_full,0.779,0.028,0.805,0.580,NaN,NaN,NaN,NaN,NaN,6.195,binary task; log1p (Init Bwd Win Byts +1)+Stan...
2,base-DT-5class,DecisionTreeClassifier,F_full,0.422,0.049,0.640,0.307,0.270,0.474,0.14,0.323,0.163,3.641,"DecisionTree, default parameters"
3,base-DT-binary,DecisionTreeClassifier,F_full,0.733,0.117,0.693,0.603,NaN,NaN,NaN,NaN,NaN,3.163,"binary task; DecisionTree, default parameters"


## 4. Bulgu
Aynı oturum gruplu katlar, F_full, varsayılan parametreler (karşılaştırma: Set B = LightGBM, F_full, `results_final.csv`):

| Deney | Macro F1 | Sağlam | Bozuk |
|---|---|---|---|
| LR, 5 sınıf | 0,424 ± 0,033 | 0,804 | 0,278 |
| DT, 5 sınıf | 0,422 ± 0,049 | 0,640 | 0,307 |
| Set B (LightGBM), 5 sınıf | 0,499 ± 0,052 | 0,837 | 0,339 |
| LR, binary | 0,779 ± 0,028 | 0,805 | 0,580 |
| DT, binary | 0,733 ± 0,117 | 0,693 | 0,603 |
| binary_B (LightGBM) | 0,817 ± 0,058 | 0,840 | 0,653 |

- Bu katlarda iki klasik model de 5 sınıfta ve binary'de LightGBM'in altında kaldı.
- 5 sınıfta LR ve DT'nin ortalaması yakın, ama hata yerleri farklı: LR sağlam alt kümede yüksek (0,804), Bot recall'u 0,000; DT bozuk alt kümede biraz yüksek (0,307), Bot recall'u 0,140, ama Dos satırlarının çoğunu Benign tahmin ediyor (sağlam 0,640).
- LR'ye F_full vermek, WP-D3'teki F_timing LR'ye (0,352 ± 0,057) göre ortalamayı yükseltti; artış çoğunlukla sağlam alt kümede (0,581 → 0,804).
- DT binary'nin std'si yüksek (0,117; katlar 0,715 / 0,858 / 0,626): skor hangi oturumların testte olduğuna bağlı.
- #7'deki eski skorlar (0,63–0,68) rastgele train/test bölmesinden geliyordu; bu tabloyla karşılaştırılamaz.